# 06 - Forecasting and verification

Purpose: run the model the way the backend does, then score its output the way
`VerificationService` does.

## The verification contract

1. A forecast is issued at a base time and targets a later period.
2. Nothing is scored until the target period has elapsed plus a grace period.
3. Scoring compares the forecast against an observation that became available
   *after* the target time, never against data already known at issue time.
4. A forecast with no matching observation stays pending; it is not scored and not
   silently dropped.

Both the issue timestamp and the observation timestamp are stored, so the temporal
ordering can be audited after the fact.

In [ ]:
import sys
from pathlib import Path

# The notebook kernel runs with the notebook directory as the working directory,
# but paths are resolved from the repository root so cells behave the same when
# executed by nbconvert, JupyterLab or an IDE.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "ml" / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

ML_SRC = REPO_ROOT / "ml" / "src"
if str(ML_SRC) not in sys.path:
    sys.path.insert(0, str(ML_SRC))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
plt.style.use("seaborn-v0_8-whitegrid")

DATA_DIR = REPO_ROOT / "data" / "processed"
ARTIFACT_ROOT = REPO_ROOT / "ml" / "artifacts"
CONFIG_PATH = REPO_ROOT / "ml" / "configs" / "training.yaml"
OBSERVATIONS = DATA_DIR / "observations.csv"
DISTRICTS = DATA_DIR / "districts.csv"

print("notebook environment ready")
print("repo root:", REPO_ROOT)
print("python", sys.version.split()[0])


In [ ]:
from rainml.data.loaders import load_districts, load_observations, attach_coordinates
from rainml.data.cleaning import clean_observations
from rainml.features.engineering import TARGET_COLUMN, build_training_frame
from rainml.models.registry import list_versions
from rainml.inference.predictor import RainfallPredictor
from rainml.inference.preprocessing import DistrictHistory

version = list_versions(ARTIFACT_ROOT)[-1]
predictor = RainfallPredictor(ARTIFACT_ROOT / version)
print("serving version", predictor.version)
print("features", len(predictor.features))

In [ ]:
cleaned, _ = clean_observations(
    attach_coordinates(load_observations(OBSERVATIONS), load_districts(DISTRICTS))
)
cleaned["observed_date"] = pd.to_datetime(cleaned["observed_at"]).dt.tz_localize(None).dt.normalize()
print(cleaned["observed_date"].min().date(), "..", cleaned["observed_date"].max().date())

## Serving-shaped inference

`predict_districts` takes per-district daily history and an issue date, which is the
same input the API assembles from `district_aggregates`.

In [ ]:
last_observation = cleaned["observed_date"].max()
issue_ts = last_observation - pd.Timedelta(days=1)
issue_date = issue_ts.date()

histories = []
for district_id, group in cleaned.groupby("district_id"):
    history_rows = group[group["observed_date"] < issue_ts]
    series = {
        row["observed_date"].date(): float(row["rainfall_mm"])
        for _, row in history_rows.iterrows()
    }
    histories.append(
        DistrictHistory(
            district_id=district_id,
            latitude=float(group["lat"].iloc[0]),
            longitude=float(group["lon"].iloc[0]),
            daily_rainfall_mm=series,
        )
    )

predictions = predictor.predict_districts(histories, issue_date=issue_date, horizon_days=1)
prediction_frame = pd.DataFrame([p.as_dict() for p in predictions])
prediction_frame.head(10)

## Verification of those forecasts

Each prediction is scored against the observation for the day it targeted. The issue
date is strictly before the observation date, so no future information enters the
feature build.

In [ ]:
from rainml.evaluation.metrics import evaluate, evaluate_by_group

target_date = issue_ts + pd.Timedelta(days=1)
truth = (
    cleaned[cleaned["observed_date"] == target_date]
    .set_index("district_id")["rainfall_mm"]
)
scored = prediction_frame.copy()
scored["observed_rainfall_mm"] = scored["district_id"].map(truth)
scored["error_mm"] = scored["observed_rainfall_mm"] - scored["predicted_rainfall_mm"]
scored["interval_hit"] = (
    (scored["observed_rainfall_mm"] >= scored["lower_bound_mm"])
    & (scored["observed_rainfall_mm"] <= scored["upper_bound_mm"])
)
scored[["district_id", "predicted_rainfall_mm", "observed_rainfall_mm", "error_mm", "interval_hit"]].round(3)

In [ ]:
scores = evaluate(
    scored["predicted_rainfall_mm"].to_numpy(),
    scored["observed_rainfall_mm"].to_numpy(),
    threshold_mm=15.0,
)
pd.Series(scores)

## Rolling verification

One forecast tells you little. The table below repeats the issue/verify cycle over a
window of days, which is how the verification job accumulates its sample.

In [ ]:
rows = []
available_days = set(cleaned["observed_date"])
first_issue_ts = issue_ts - pd.Timedelta(days=20)

for offset in range(21):
    day_ts = first_issue_ts + pd.Timedelta(days=offset)
    if day_ts not in available_days:
        continue
    day = day_ts.date()
    day_histories = [
        DistrictHistory(h.district_id, h.latitude, h.longitude,
                        {k: v for k, v in h.daily_rainfall_mm.items() if k < day})
        for h in histories
    ]
    day_predictions = predictor.predict_districts(day_histories, issue_date=day)
    next_day = day_ts + pd.Timedelta(days=1)
    day_truth = cleaned[cleaned["observed_date"] == next_day].set_index("district_id")["rainfall_mm"]
    for p in day_predictions:
        observed = day_truth.get(p.district_id)
        if observed is None:
            continue
        rows.append(
            {
                "issue_date": day,
                "target_date": next_day.date(),
                "district_id": p.district_id,
                "predicted": p.predicted_rainfall_mm,
                "observed": float(observed),
                "lead_hours": 24.0,
            }
        )

rolling = pd.DataFrame(rows)
print(len(rolling), "verified forecasts across", rolling["issue_date"].nunique(), "issue dates")

In [ ]:
overall = evaluate(rolling["predicted"].to_numpy(), rolling["observed"].to_numpy())
pd.Series(overall)

In [ ]:
rolling["error"] = rolling["observed"] - rolling["predicted"]
rolling["absolute_error"] = rolling["error"].abs()
daily = rolling.groupby("issue_date").agg(
    mean_forecast=("predicted", "mean"),
    mean_observed=("observed", "mean"),
    mae=("absolute_error", "mean"),
)
daily.index = pd.to_datetime(daily.index)
ax = daily.plot(figsize=(12, 4.5), marker="o")
ax.set_ylabel("rainfall (mm)")
ax.set_title("Rolling forecast versus observation")
plt.tight_layout()
plt.show()
daily.round(3)

## Skill by lead-time bucket

The API groups verification results this way, so the dashboard and this table agree.

In [ ]:
def bucket(hours: float) -> str:
    if hours <= 6:
        return "0-6h"
    if hours <= 12:
        return "6-12h"
    if hours <= 24:
        return "12-24h"
    return ">24h"


rolling["lead_bucket"] = rolling["lead_hours"].map(bucket)
pd.DataFrame(
    evaluate_by_group(
        rolling.rename(columns={"predicted": "prediction"}),
        observed_column="observed",
        group_column="lead_bucket",
    )
).T[
    ["sample_size", "mae_mm", "rmse_mm", "csi"]
].round(4)

## Temporal ordering audit

The invariant the whole verification design depends on: for every verified row, the
issue date is strictly before the observation date. A single violation would mean a
future observation was used to build a feature or to score a forecast.

In [ ]:
violations = (rolling["issue_date"] >= rolling["target_date"]).sum()
print("rows:", len(rolling))
print("ordering violations:", violations)
assert violations == 0
print("temporal ordering verified")

## Registry state

The database holds the promotion lifecycle. In PostgreSQL:

```sql
SELECT version, status, activated_at FROM model_versions ORDER BY created_at DESC;
SELECT batch_uid, forecasts_verified, status FROM verification_batches ORDER BY started_at DESC LIMIT 5;
```

A new version is created as `candidate`. Promotion to `active` happens only through the
promotion endpoint, and only when the policy agrees the candidate is better.

In [ ]:
from rainml.models.registry import list_versions, load_metadata, verify_checksum

for v in list_versions(ARTIFACT_ROOT):
    meta = load_metadata(ARTIFACT_ROOT, v)
    print(
        v,
        meta["status"],
        meta["training_data_range"],
        "checksum ok" if verify_checksum(ARTIFACT_ROOT, v) else "CHECKSUM MISMATCH",
    )